# Falcon production checkpoint/resume gate

This is a bounded infrastructure test, not a quality run. It verifies the current pushed trainer after sampler-state hardening: local step 2 -> 4 resume, exact persisted checkpoint retrieval, and resume from the retrieved checkpoint to step 6 with an evaluation boundary. No candidate is promoted.


In [ ]:
import json, os, platform, shutil, subprocess, time
from pathlib import Path
WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
DATA = REPO / 'experiments' / 'falcon-resume-gate' / 'data'
RUN = REPO / 'experiments' / 'falcon-resume-gate' / 'local-resume'
PERSIST_DATASET = 'toheebogunade/jamii-afya-falcon-production-checkpoints'

def run(command, cwd=REPO, env=None):
    merged = os.environ.copy(); merged.update(env or {}); merged['PYTHONUNBUFFERED'] = '1'
    print('STREAM', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=cwd, env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(f'[{time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}] {line}', end='', flush=True)
    code = process.wait()
    print('EXIT', code, flush=True)
    if code: raise RuntimeError(f'command failed with exit {code}: {command}')

if not REPO.exists():
    run(['git', 'clone', '--depth', '1', '--branch', 'research/edge35-adaptive-streaming', 'https://github.com/qeinstein/adtc-llm-limited-hardware.git', str(REPO)], cwd=WORK)
else:
    run(['git', 'fetch', 'origin', 'research/edge35-adaptive-streaming'], cwd=REPO)
    run(['git', 'checkout', '-B', 'research/edge35-adaptive-streaming', 'origin/research/edge35-adaptive-streaming'], cwd=REPO)
run([os.sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-falcon-production.txt'])
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True, capture_output=True, check=False).stdout.strip()
print(json.dumps({'gpu': gpu, 'python': platform.python_version()}, indent=2), flush=True)
if 'P100' in gpu:
    run([os.sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy<2'])
    run([os.sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu118'])
    run([os.sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao', 'torchvision', 'torchaudio', 'bitsandbytes'])
    run([os.sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-deps', 'transformers==4.53.3', 'tokenizers==0.21.4', 'peft==0.15.2', 'accelerate==1.7.0'])
run([os.sys.executable, '-c', 'import torch, transformers, peft; print(torch.__version__, torch.cuda.get_device_capability(0), transformers.__version__, peft.__version__)'])

if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True, exist_ok=True)
mcqa = REPO / 'output' / 'accuracy_sft.jsonl'
if mcqa.exists(): mcqa.unlink()
run([os.sys.executable, '-u', 'scripts/build_accuracy_sft.py', '--datasets', 'arc_easy', 'medmcqa', '--max-per-dataset', '4', '--letter-permutations', '1', '--seed', '3407', '--fail-on-source-error', '--out', str(mcqa)])
run([os.sys.executable, '-u', 'scripts/build_falcon_dataset.py', '--config', 'configs/falcon-production-v1.json', '--out-dir', str(DATA)])

base = [os.sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', 'configs/falcon-production-v1.json', '--data-dir', str(DATA), '--stage', 'stage_a_capability_preserving', '--allow-ephemeral', '--quantize', 'none', '--compute-dtype', 'fp16']
if RUN.exists(): shutil.rmtree(RUN)
run(base + ['--run-dir', str(RUN), '--max-steps', '2', '--save-steps', '1', '--eval-steps', '2'])
run(base + ['--run-dir', str(RUN), '--max-steps', '4', '--save-steps', '1', '--eval-steps', '2', '--resume-from-checkpoint', 'latest'])
stage = RUN / 'stage_a_capability_preserving'
checkpoints = stage / 'checkpoints'
first = checkpoints / 'checkpoint-2'
second = checkpoints / 'checkpoint-4'
assert json.loads((first / 'trainer_state.json').read_text())['global_step'] == 2
assert json.loads((second / 'trainer_state.json').read_text())['global_step'] == 4
for checkpoint in (first, second):
    manifest = json.loads((checkpoint / 'checkpoint_manifest.json').read_text())
    assert manifest['complete'] is True and manifest['sampler_required'] is True
    assert (checkpoint / 'sampler_state.json').is_file()
    assert (checkpoint / 'optimizer.pt').is_file() and (checkpoint / 'scheduler.pt').is_file() and (checkpoint / 'rng_state.pth').is_file()

run([os.sys.executable, '-u', 'scripts/persist_checkpoint.py', '--checkpoint', str(second), '--dataset', PERSIST_DATASET, '--message', 'Falcon sampler-aware resume gate'], env={'FALCON_CHECKPOINT_DATASET': PERSIST_DATASET})
retrieved = REPO / 'experiments' / 'falcon-resume-gate' / 'retrieved'
if retrieved.exists(): shutil.rmtree(retrieved)
run([os.sys.executable, '-u', 'scripts/verify_persisted_checkpoint.py', '--dataset', PERSIST_DATASET, '--out-dir', str(retrieved)], env={'FALCON_CHECKPOINT_DATASET': PERSIST_DATASET})
remote_states = sorted(retrieved.rglob('trainer_state.json'), key=lambda p: json.loads(p.read_text()).get('global_step', -1))
remote_step4 = next(path.parent for path in remote_states if json.loads(path.read_text()).get('global_step') == 4)
remote_manifest = json.loads((remote_step4 / 'checkpoint_manifest.json').read_text())
assert remote_manifest['sampler_required'] is True and (remote_step4 / 'sampler_state.json').is_file()
remote_run = REPO / 'experiments' / 'falcon-resume-gate' / 'remote-resume'
run(base + ['--run-dir', str(remote_run), '--max-steps', '6', '--save-steps', '1', '--eval-steps', '2', '--resume-from-checkpoint', str(remote_step4)])
remote_resumed = remote_run / 'stage_a_capability_preserving' / 'checkpoints' / 'checkpoint-6' / 'trainer_state.json'
assert json.loads(remote_resumed.read_text())['global_step'] == 6
result = {'status': 'PASS', 'repo_sha': subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True, capture_output=True, check=True).stdout.strip(), 'local_steps': [2, 4], 'retrieved_step': 4, 'remote_resumed_step': 6, 'sampler_state_verified': True, 'optimizer_scheduler_rng_verified': True, 'persistence_dataset': PERSIST_DATASET}
(WORK / 'falcon-resume-gate-result.json').write_text(json.dumps(result, indent=2) + '\n')
print(json.dumps(result, indent=2), flush=True)
